# Policy Document Conversion with VLM Artifact Interpretation

This notebook demonstrates high-fidelity conversion of **policy documents** containing non-text artifacts (risk matrices, decision trees, compliance charts, forms, screenshots) into structured Markdown on OpenShift AI.

The approach combines:
1. **Docling standard pipeline** for structural document parsing (headings, tables, numbered clauses, reading order)
2. **VLM (Vision Language Model)** for classifying and interpreting extracted images with artifact-type-specific prompts
3. **Assembly** that replaces image references with structured Markdown interpretations wrapped in artifact-type markers

This is the step-by-step walkthrough of the [`docling-vlm-policy` KFP pipeline](../../kubeflow-pipelines/docling-vlm-policy/).

## 1. Installation

Install required packages. Run once per session.

In [ ]:
!pip install -qq docling requests

## 2. Deploy the VLM InferenceService

The VLM must be deployed as a KServe InferenceService before running the pipeline. The manifest is provided at `kubeflow-pipelines/docling-vlm-policy/kserve/vlm-inferenceservice.yaml`.

```bash
# Deploy Granite Vision via vLLM on KServe
kubectl apply -f ../../kubeflow-pipelines/docling-vlm-policy/kserve/vlm-inferenceservice.yaml

# Wait for it to be ready
kubectl get inferenceservice policy-vlm -w
```

Once ready, the endpoint is accessible in-cluster at:
```
http://policy-vlm-predictor.<namespace>.svc.cluster.local:8080/v1/chat/completions
```

## 3. Configuration

### Set the VLM endpoint

Point to your deployed VLM InferenceService. When running on OpenShift AI, use the in-cluster service URL.

In [ ]:
import os

VLM_ENDPOINT_URL = os.getenv(
    "VLM_SERVICE_ENDPOINT_URL",
    "http://policy-vlm-predictor.default.svc.cluster.local:8080/v1/chat/completions",
)
VLM_MODEL_NAME = os.getenv("VLM_SERVICE_MODEL_NAME", "ibm-granite/granite-vision-3.2-2b")
VLM_API_KEY = os.getenv("VLM_SERVICE_API_KEY", "")
VLM_MAX_TOKENS = 1024
VLM_TIMEOUT = 120

### Set input documents

For testing, use sample PDFs from the Docling test suite. For real policy documents, point to your own files or the policy examples in [`asago-ai/asago-examples`](https://github.com/asago-ai/asago-examples) under `asago-policy-mapper/policy_examples/`.

In [ ]:
from pathlib import Path

files = [
    "https://raw.githubusercontent.com/py-pdf/sample-files/refs/heads/main/003-pdflatex-image/pdflatex-image.pdf",
]

output_dir = Path("policy-conversion-output")
output_dir.mkdir(parents=True, exist_ok=True)

## 4. Convert Documents with Docling

Use the standard Docling pipeline for structural parsing. We set `image_mode=ImageRefMode.REFERENCED` so that images are exported as separate PNG files alongside the Markdown, which the VLM interpretation step needs.

In [ ]:
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.pipeline.standard_pdf_pipeline import StandardPdfPipeline
from docling_core.types.doc import ImageRefMode

pipeline_options = PdfPipelineOptions()
pipeline_options.generate_page_images = True
pipeline_options.do_table_structure = True
pipeline_options.do_ocr = True
pipeline_options.document_timeout = 600
pipeline_options.accelerator_options = AcceleratorOptions(
    num_threads=4, device=AcceleratorDevice.AUTO
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options,
            pipeline_cls=StandardPdfPipeline,
        )
    }
)

results = {}
for file in files:
    print(f"Converting {file}...")
    result = converter.convert(file)
    doc = result.document
    stem = Path(file).stem

    # Save JSON (DoclingDocument)
    json_path = output_dir / f"{stem}.json"
    doc.save_as_json(json_path, image_mode=ImageRefMode.REFERENCED)
    print(f"  Saved JSON: {json_path}")

    # Save Markdown with referenced images
    md_path = output_dir / f"{stem}.md"
    doc.save_as_markdown(md_path, image_mode=ImageRefMode.REFERENCED)
    print(f"  Saved Markdown: {md_path}")

    results[stem] = {"document": doc, "md_path": md_path, "json_path": json_path}

print(f"\nConverted {len(results)} document(s)")

## 5. Find Extracted Images

Inspect the Markdown output to find image references that need VLM interpretation.

In [ ]:
import re

IMG_PATTERN = re.compile(r"!\[([^\]]*)\]\(([^)]+)\)")

for stem, info in results.items():
    md_content = info["md_path"].read_text(encoding="utf-8")
    matches = list(IMG_PATTERN.finditer(md_content))
    print(f"{stem}: {len(matches)} image(s) found")
    for i, m in enumerate(matches):
        print(f"  [{i}] {m.group(2)}")

## 6. VLM Helper Function

Define the helper to call the VLM via the OpenAI-compatible chat completions API with an image.

In [ ]:
import base64
import requests


def call_vlm(image_b64: str, prompt: str, max_tokens: int = VLM_MAX_TOKENS) -> str:
    """Send an image + prompt to the VLM and return the text response."""
    payload = {
        "model": VLM_MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{image_b64}"
                        },
                    },
                ],
            }
        ],
        "max_tokens": max_tokens,
    }
    headers = {}
    if VLM_API_KEY:
        headers["Authorization"] = f"Bearer {VLM_API_KEY}"

    resp = requests.post(VLM_ENDPOINT_URL, json=payload, headers=headers, timeout=VLM_TIMEOUT)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]

## 7. Classify and Interpret Artifacts

For each image extracted by Docling:
1. **Classify** it into an artifact type (risk matrix, decision tree, chart, form, screenshot, diagram, other)
2. **Interpret** it using a type-specific VLM prompt that produces structured Markdown

In [ ]:
CLASSIFICATION_PROMPT = (
    "You are analyzing an image extracted from a policy document. "
    "Classify this image into exactly one of the following categories. "
    "Respond with ONLY the category name, nothing else.\n\n"
    "Categories:\n"
    "- chart: Any type of chart, graph, or plot\n"
    "- risk_matrix: A risk assessment matrix or heat map\n"
    "- decision_tree: A decision tree or flowchart with branches\n"
    "- form: A form, questionnaire, or template\n"
    "- screenshot: A screenshot of a user interface\n"
    "- diagram: An architecture diagram, process flow, or org chart\n"
    "- other: Any other type of image"
)

INTERPRETATION_PROMPTS = {
    "risk_matrix": (
        "Convert this risk matrix into structured Markdown. "
        "Create a Markdown table that preserves all row headers, column headers, "
        "and cell values. Include severity levels and any color coding described as "
        "text labels (e.g., 'High Risk'). Add any legends or footnotes below the table."
    ),
    "decision_tree": (
        "Convert this decision tree into a nested Markdown list representing the "
        "complete decision logic. Use indentation to show hierarchy and branches. "
        "Preserve all labels, conditions, and outcomes exactly as shown."
    ),
    "chart": (
        "Describe this chart in structured Markdown. Include chart type, title, "
        "axis labels, a data summary table with key values, and notable trends."
    ),
    "form": (
        "Convert this form into a Markdown table with 'Field' and 'Value/Description' "
        "columns. Include all field labels, values, checkboxes, and instructions."
    ),
    "screenshot": (
        "Describe this UI screenshot as a structured step-by-step walkthrough. "
        "Identify the application, describe visible elements, and note the current state."
    ),
    "diagram": (
        "Describe this diagram in structured Markdown. List all components, "
        "their relationships, directional flows, and any annotations."
    ),
    "other": (
        "Describe the content of this image in structured Markdown. "
        "Preserve all visible text, numbers, and structural information."
    ),
}

VALID_TYPES = set(INTERPRETATION_PROMPTS.keys())

In [ ]:
import json

all_artifacts = {}

for stem, info in results.items():
    md_content = info["md_path"].read_text(encoding="utf-8")
    matches = list(IMG_PATTERN.finditer(md_content))

    if not matches:
        print(f"{stem}: no images, skipping")
        all_artifacts[stem] = []
        continue

    artifacts = []
    for idx, match in enumerate(matches):
        img_ref = match.group(2)

        # Resolve image path
        img_path = output_dir / img_ref
        if not img_path.exists():
            img_path = output_dir / Path(img_ref).name
        if not img_path.exists():
            candidates = list(output_dir.rglob(Path(img_ref).name))
            img_path = candidates[0] if candidates else None

        if img_path is None or not img_path.exists():
            print(f"  [{idx}] Image not found: {img_ref}")
            continue

        with open(img_path, "rb") as f:
            img_b64 = base64.b64encode(f.read()).decode("utf-8")

        # Classify
        print(f"  [{idx}] Classifying...")
        raw_type = call_vlm(img_b64, CLASSIFICATION_PROMPT, max_tokens=20)
        artifact_type = raw_type.strip().lower().replace(" ", "_")
        if artifact_type not in VALID_TYPES:
            artifact_type = "other"
        print(f"  [{idx}] Type: {artifact_type}")

        # Interpret
        print(f"  [{idx}] Interpreting as {artifact_type}...")
        interpretation = call_vlm(img_b64, INTERPRETATION_PROMPTS[artifact_type])
        print(f"  [{idx}] Done ({len(interpretation)} chars)")

        artifacts.append({
            "index": idx,
            "type": artifact_type,
            "source_image": str(img_ref),
            "match_text": match.group(0),
            "interpretation": interpretation.strip(),
        })

    all_artifacts[stem] = artifacts
    print(f"{stem}: processed {len(artifacts)} artifact(s)")

## 8. Assemble Policy Markdown

Replace each image reference in the Markdown with the VLM-generated interpretation, wrapped in `<!-- artifact: {type} -->` / `<!-- /artifact -->` markers.

In [ ]:
for stem, info in results.items():
    md_content = info["md_path"].read_text(encoding="utf-8")
    artifacts = all_artifacts.get(stem, [])

    for artifact in artifacts:
        replacement = (
            f"<!-- artifact: {artifact['type']} -->\n"
            f"{artifact['interpretation']}\n"
            f"<!-- /artifact -->"
        )
        md_content = md_content.replace(artifact["match_text"], replacement, 1)

    # Save assembled policy markdown
    policy_path = output_dir / f"{stem}_policy.md"
    policy_path.write_text(md_content, encoding="utf-8")
    print(f"Saved: {policy_path}")

    # Save artifact metadata
    meta_path = output_dir / f"{stem}_artifacts.json"
    with open(meta_path, "w") as f:
        json.dump(
            {"document": stem, "total_artifacts": len(artifacts), "artifacts": artifacts},
            f, indent=2, ensure_ascii=False,
        )
    print(f"Saved: {meta_path}")

## 9. Review Output

Inspect the assembled Markdown to verify artifact interpretations.

In [ ]:
from IPython.display import Markdown, display

for stem in results:
    policy_path = output_dir / f"{stem}_policy.md"
    if policy_path.exists():
        content = policy_path.read_text(encoding="utf-8")
        display(Markdown(f"### {stem}_policy.md\n\n---"))
        display(Markdown(content[:3000]))  # Show first 3000 chars
        if len(content) > 3000:
            print(f"... ({len(content) - 3000} more characters)")

## 10. Quality Evaluation

Basic quality checks on the assembled output.

In [ ]:
for stem in results:
    policy_path = output_dir / f"{stem}_policy.md"
    meta_path = output_dir / f"{stem}_artifacts.json"

    if not policy_path.exists():
        continue

    content = policy_path.read_text(encoding="utf-8")
    with open(meta_path) as f:
        meta = json.load(f)

    # Count artifact markers
    open_markers = content.count("<!-- artifact:")
    close_markers = content.count("<!-- /artifact -->")

    # Check for remaining image references (should be zero)
    remaining_imgs = len(IMG_PATTERN.findall(content))

    # Check structural elements
    headings = len(re.findall(r"^#+\s", content, re.MULTILINE))
    tables = content.count("|---")

    print(f"=== {stem} ===")
    print(f"  Total artifacts interpreted: {meta['total_artifacts']}")
    print(f"  Artifact markers (open/close): {open_markers}/{close_markers}")
    print(f"  Remaining unprocessed images: {remaining_imgs}")
    print(f"  Headings preserved: {headings}")
    print(f"  Tables in output: {tables}")
    print(f"  Output length: {len(content)} chars")
    print()

    # Type distribution
    type_counts = {}
    for a in meta["artifacts"]:
        type_counts[a["type"]] = type_counts.get(a["type"], 0) + 1
    if type_counts:
        print(f"  Artifact types: {type_counts}")

## Additional Resources

- [Pipeline README](../../kubeflow-pipelines/docling-vlm-policy/README.md) — full architecture and deployment guide
- [KServe InferenceService manifest](../../kubeflow-pipelines/docling-vlm-policy/kserve/vlm-inferenceservice.yaml) — VLM deployment
- [Sample policy documents](https://github.com/asago-ai/asago-examples) — `asago-policy-mapper/policy_examples/`
- [Docling documentation](https://docling-project.github.io/docling/) — conversion pipeline reference

For feedback, please [open an issue](https://github.com/opendatahub-io/odh-data-processing/issues).